[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/adaptive-data/configure_adaptive_data.ipynb)

# Configure Adaptive Data

Import a dataset, map its columns, apply recipes and brand controls, then download an adapted dataset ready for training — with a cost estimate before anything runs.

**Docs:** [Configure Adaptive Data](https://docs.adaptionlabs.ai/adaptive-data/configure-adaptive-data/) · [quickstart](https://docs.adaptionlabs.ai/adaptive-data-quickstart/)  
**Blog:** [Adaptive Data: Designed for Change](https://adaptionlabs.ai/blog/adaption-launches-adaptive-data-beta)  
**Prerequisites:** an Adaption API key ([create one](https://adaptionlabs.ai/app/settings?tab=api_keys)) set as `ADAPTION_API_KEY` (env var or Colab secret).

## Overview

Adaptive Data treats your data space as **dynamic and fully malleable** rather than a fixed asset you assemble once and freeze. It brings data-optimization techniques typically reserved for frontier labs to every builder — an average **82% improvement in data quality** across early deployments, with support for **242 languages**.

This runbook configures that workflow end to end in code: **import** a dataset, **map** its columns, apply **recipes** and **brand controls**, **estimate** the cost before anything runs, **adapt**, **check quality**, and **download** a training-ready dataset.

Background: [Adaptive Data: Designed for Change](https://adaptionlabs.ai/blog/adaption-launches-adaptive-data-beta).

In [ ]:
%pip install -q adaption pandas pyarrow

## Setup

Load your API key. In Colab, add `ADAPTION_API_KEY` to the secrets panel (🔑 in the sidebar) and enable it for this notebook. Running locally, the cell prompts for the key if the environment variable is not set.

In [ ]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Parameters

Everything you might change lives here, so the rest of the notebook runs unedited.

In [ ]:
# Source dataset to import from Hugging Face (Alpaca is a small instruction set).
DATASET_URL = "https://huggingface.co/datasets/tatsu-lab/alpaca"
DATASET_FILES = ["data/train-00000-of-00001-a09b74b3ef9c3b56.parquet"]

# Map source columns to the roles Adaptive Data expects.
COLUMN_MAPPING = {
    "prompt": "instruction",
    "completion": "output",
    "context": ["input"],
}

# Keep runs cheap while you iterate. Set to None to process the whole dataset.
MAX_ROWS = 200

# "instruction_dataset" for supervised fine-tuning, or "preference_pairs" for DPO.
TRAINING_TYPE = "instruction_dataset"

## Step 1 — Import a dataset

[`datasets.create`](https://docs.adaptionlabs.ai/adaptive-data/create-a-dataset/) pulls the file(s) from Hugging Face and returns a `dataset_id`. You can also `client.datasets.upload_file("local.csv")` or import from Kaggle.

In [ ]:
dataset = client.datasets.create(
    source={"url": DATASET_URL, "files": DATASET_FILES},
)
dataset_id = dataset.dataset_id
print("dataset_id:", dataset_id)

Imports run asynchronously. Poll the status until the row count is populated before adapting. The loop fails loudly if ingestion fails.

In [ ]:
import time

while True:
    status = client.datasets.get_status(dataset_id)
    if status.status == "failed":
        err = status.error_data
        msg = (err and err.message) or "unknown error"
        raise RuntimeError(f"Ingestion failed: {msg}")
    if status.row_count is not None:
        print(f"Imported {status.row_count} rows")
        break
    time.sleep(5)

## Step 2 — Choose recipes and brand controls

A run always applies settings in the same order, so argument order does not matter. **Recipes** transform the mapped data; **brand controls** shape the generated completions.

- `deduplication` removes duplicate prompts.
- `prompt_rephrase` rewrites prompts for clarity (not applied to `chat` mappings).
- `reasoning_traces` saves the reasoning behind each completion — turn it on for chain-of-thought models.
- `length` controls verbosity: `minimal`, `concise`, `detailed`, or `extensive`.
- `hallucination_mitigation` grounds fact-sensitive answers with web search.
- `blueprint` is a freeform system instruction for tone, persona, or wording.

In [ ]:
recipe_specification = {
    "recipes": {
        "deduplication": True,
        "prompt_rephrase": True,
        "reasoning_traces": False,
    },
}

brand_controls = {
    "length": "concise",
    "hallucination_mitigation": False,
    "blueprint": "Answer in clear, friendly English with a direct tone.",
}

job_specification = {"max_rows": MAX_ROWS} if MAX_ROWS else {}

## Step 3 — Estimate the run

Pass `estimate=True` to validate the exact configuration and get a credit quote **without** starting a run. Credits scale with the rows processed plus any rows that language expansion would add.

In [ ]:
estimate = client.datasets.adapt(
    dataset_id,
    column_mapping=COLUMN_MAPPING,
    training_type=TRAINING_TYPE,
    recipe_specification=recipe_specification,
    brand_controls=brand_controls,
    job_specification=job_specification,
    estimate=True,
)
print(f"Estimated credits: {estimate.estimated_credits_consumed}")

## Step 4 — Run the adaptation

Submit the same configuration without `estimate=True` to start the run.

In [ ]:
run = client.datasets.adapt(
    dataset_id,
    column_mapping=COLUMN_MAPPING,
    training_type=TRAINING_TYPE,
    recipe_specification=recipe_specification,
    brand_controls=brand_controls,
    job_specification=job_specification,
)
print(f"Run started: {run.run_id}")

> **Want preference pairs for DPO?** Set `TRAINING_TYPE = "preference_pairs"` in the parameters cell and re-run Steps 3–4. Adaptive Data then generates a chosen/rejected pair per row instead of a single completion. The same recipes, brand controls, and row limits apply.

## Step 5 — Wait for completion

`wait_for_completion` polls until the adaptation run finishes.

In [ ]:
status = client.datasets.wait_for_completion(dataset_id, timeout=3600)
if status.status == "failed":
    raise RuntimeError(f"Adaptive Data failed: {status.error_data}")
print("Adaptation status:", status.status)

## Step 6 — Check quality

Evaluation runs on its own schedule, so it can finish after adaptation. Poll with a deadline so the notebook cannot wait forever. Scores are on a 0–10 scale.

In [ ]:
deadline = time.monotonic() + 600
while True:
    evaluation = client.datasets.get_evaluation(dataset_id)
    print("Evaluation status:", evaluation.status)
    if evaluation.status in ("succeeded", "failed", "skipped"):
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Evaluation did not finish within 10 minutes")
    time.sleep(5)

if evaluation.status == "succeeded" and evaluation.quality:
    q = evaluation.quality
    print(f"Score before: {q.score_before}")
    print(f"Score after:  {q.score_after}")
    print(f"Improvement:  {q.improvement_percent}%")

## Step 7 — Download the adapted dataset

`download` streams the processed rows. `file_format="parquet"` returns a **gzipped tar of Parquet shards** (`.tar.gz`), not a single file — extract it before reading. Use `csv`, `json`, or `jsonl` for plain text.

In [ ]:
client.datasets.download(dataset_id, file_format="parquet").write_to_file(
    "adapted-dataset.tar.gz"
)

import tarfile

with tarfile.open("adapted-dataset.tar.gz") as archive:
    archive.extractall("adapted-dataset/")
print("Extracted to adapted-dataset/")

In [ ]:
import glob

import pandas as pd

shard = sorted(glob.glob("adapted-dataset/**/*.parquet", recursive=True))[0]
pd.read_parquet(shard).head()

## Next steps

Adaptive Data is **designed for change** — come back and re-adapt as your data and goals evolve:

- [Train a model with AutoScientist](https://docs.adaptionlabs.ai/autoscientist-quickstart/) on this adapted dataset.
- [Combine datasets](https://docs.adaptionlabs.ai/adaptive-data/combine-datasets/) if related examples live in several datasets.
- [Evaluate dataset quality](https://docs.adaptionlabs.ai/adaptive-data/evaluate-dataset-quality/) in more depth.

## Clean up

The adapted dataset stays in your account until you delete it. Uncomment to remove it.

In [ ]:
# client.datasets.delete(dataset_id=dataset_id)